Production Style Monitoring: common metrics
| Metric                     | Why it matters                                             |
| -------------------------- | ---------------------------------------------------------- |
| Request count (RPS/QPS)    | How much traffic is the system handling?                   |
| Success rate               | Percentage of successful requests.                         |
| Error rate                 | Timeouts, API failures, parsing errors.                    |
| Token usage                | Prompt tokens, completion tokens, total tokens.            |
| Cost                       | Cost per request and total spend.                          |
| Throughput                 | Requests processed per second or tokens per second.        |
| Queue time                 | Time waiting before processing starts.                     |
| Model latency              | Time spent waiting for the LLM response.                   |
| Retrieval latency          | Time spent querying the vector database.                   |
| Tool latency               | Time spent calling external APIs or tools.                 |
| End-to-end latency         | Total user-perceived response time.                        |
| Cache hit rate             | Percentage of requests served from cache.                  |
| Hallucination rate         | Fraction of responses judged incorrect or unsupported.     |
| User satisfaction          | Average feedback score or thumbs-up ratio.                 |
| Retry rate                 | How often requests need retries due to transient failures. |
| Timeout rate               | Frequency of requests exceeding a timeout threshold.       |
| Prompt version performance | Compare metrics across prompt revisions.                   |
| Model comparison           | Compare latency, cost, and quality between models.         |
| Token efficiency           | Output quality relative to token consumption.              |


#!pip install langchain langchain-core langchain_community langchain_openai
#Using langchain for templates
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

In [1]:
#Check versions
import os
import transformers
import numpy as np
import tokenizers
import sys
import transformers.models.t5
import langchain_huggingface
print("transformers",transformers.__version__)
print("numpy",np.__version__)
print(transformers.__file__)
print(transformers.models.t5.__file__)
print("Python:", sys.executable)
print("Transformers path:", transformers.__file__)
print("Tokenizers:", tokenizers.__version__)
print("Tokenizers path:", tokenizers.__file__)
from importlib.metadata import version
print(version("langchain-huggingface"))
!pip show torch
!pip show torchvision
!pip show torchaudio

transformers 4.55.2
numpy 1.26.4
e:\Lesson_2_demos\venv\lib\site-packages\transformers\__init__.py
e:\Lesson_2_demos\venv\lib\site-packages\transformers\models\t5\__init__.py
Python: e:\Lesson_2_demos\venv\Scripts\python.exe
Transformers path: e:\Lesson_2_demos\venv\lib\site-packages\transformers\__init__.py
Tokenizers: 0.21.4
Tokenizers path: e:\Lesson_2_demos\venv\lib\site-packages\tokenizers\__init__.py
1.2.2
Name: torch
Version: 2.2.2
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: e:\lesson_2_demos\venv\lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, typing-extensions
Required-by: accelerate, peft, sentence-transformers, torchvision
Name: torchvision
Version: 0.17.2
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-e

In [2]:
#Using different models 
#(may throw error due to tokenizer being new version which came in when we installed other packages)
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

In [3]:
#downgrade protobuf to avoid warnings,if needed
#!pip install --upgrade protobuf==4.25.3
#restart session,if above step done
#!pip show protobuf

In [4]:
#Using smaller model than xl
#Online Mode
#generator = pipeline("text2text-generation", model="google/flan-t5-large")

#Offline Mode (to avoid redownloading or checking latest on web)
model_name = "google/flan-t5-large"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
)


Device set to use cpu


In [5]:
#WE can create generator with more details specified

# generator = pipeline(
#     "text2text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=100, 
#     torch_dtype=torch.float32,  # Uses lower precision for efficiency
#     device=0 if torch.cuda.is_available() else -1  # Use GPU if available
# )

In [6]:
prompt = "What is crypto currency"
response = generator(prompt)
print(response)

[{'generated_text': 'crypto currency'}]


In [7]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

template2 = "Please write a {length} review,of the book {book_title}. "
#template2 = "Summarize the book {book_title} in {length} words. "
input_variables2 = [ "length", "book_title" ]
prompt = PromptTemplate(
    input_variables=input_variables2,
    template=template2
)
#To check
#formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
#print(formatted_prompt)
llm = HuggingFacePipeline(pipeline=generator)

#Chaining
chain_new = prompt | llm
response1 = chain_new.invoke({
    "length": "short",
    "book_title": "House of Dragon"
})
print('response1-chain: ',response1)

#Passing prompt directly into Pipeline
prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)
response2 = generator(prompt)
print('response2-direct: ', response2[0]["generated_text"])

#Passing prompt into Function
prompt = template2.format(
    length="short",
    book_title="Sherlock Holmes"
)

response1-chain:  's a spooky tale of dragons , witches , and witchcraft . . .
response2-direct:  's a spooky tale of dragons , witches , and witchcraft . . .


In [8]:
#Monitoring Local LLM
# ----------------------------------------------------
# Production Metric 1 : End-to-End Latency
#
# Measures the total time taken to process one request.
# This is the response time experienced by the user.
# ----------------------------------------------------

import time

prompt = template2.format(
    length="short",
    book_title="Sherlock Holmes"
)

start = time.perf_counter()

response = generator(prompt)

end = time.perf_counter()

latency_ms = (end-start)*1000

print(response[0]["generated_text"])
print(f"\nLatency : {latency_ms:.2f} ms")

Sherlock Holmes is a witty, witty, witty novel about a man who is a master of his craft . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . .

Latency : 33567.23 ms


In [9]:
# ----------------------------------------------------
# Production Metric 2 : Average Latency
#
# Runs the same workload multiple times and computes
# the average response time.
# ----------------------------------------------------

import statistics
import time

latencies=[]

for i in range(5):

    start=time.perf_counter()

    generator(prompt)

    end=time.perf_counter()

    latencies.append((end-start)*1000)

print(latencies)

print(f"\nAverage Latency : {statistics.mean(latencies):.2f} ms")

[35480.14790005982, 32283.83140009828, 32224.489300046116, 35304.093500133604, 37842.67030004412]

Average Latency : 34627.05 ms


In [10]:
# ----------------------------------------------------
# Production Metric 3 : P95 Latency
#
# Indicates the latency below which 95% of requests
# complete.
#
# Helps identify slow requests that the average
# latency may hide.
# ----------------------------------------------------

latencies.sort()

index=min(int(len(latencies)*0.95),len(latencies)-1)

print("Sorted Latencies")

print(latencies)

print(f"P95 Latency : {latencies[index]:.2f} ms")

Sorted Latencies
[32224.489300046116, 32283.83140009828, 35304.093500133604, 35480.14790005982, 37842.67030004412]
P95 Latency : 37842.67 ms


In [11]:
# ----------------------------------------------------
# Production Metric 4 : Throughput
#
# Measures how many requests can be processed
# every second.
# ----------------------------------------------------

requests=20

start=time.perf_counter()

for i in range(requests):

    generator(prompt)

elapsed=time.perf_counter()-start

throughput=requests/elapsed

print(f"Processed : {requests}")

print(f"Elapsed : {elapsed:.2f} sec")

print(f"Throughput : {throughput:.2f} requests/sec")

Processed : 20
Elapsed : 733.57 sec
Throughput : 0.03 requests/sec


In [12]:
# ----------------------------------------------------
# Production Metric 5 : Success Rate
#
# Percentage of requests completed successfully.
# ----------------------------------------------------

success=0
failure=0

for i in range(10):

    try:

        generator(prompt)

        success+=1

    except Exception as e:

        failure+=1

        print(e)

total=success+failure

print(f"Successful : {success}")
print(f"Failed : {failure}")
print(f"Success Rate : {(success/total)*100:.2f}%")

Successful : 10
Failed : 0
Success Rate : 100.00%


In [13]:
# ----------------------------------------------------
# Production Metric 6 : Error Rate
#
# Percentage of failed requests.
# ----------------------------------------------------

print(f"Error Rate : {(failure/total)*100:.2f}%")

Error Rate : 0.00%


In [14]:
# ----------------------------------------------------
# Production Metric 7 : Requests Per Minute
#
# Measures application traffic.
# ----------------------------------------------------

requests=30

start=time.perf_counter()

for i in range(requests):

    generator(prompt)

elapsed=time.perf_counter()-start

rpm=requests/(elapsed/60)

print(f"Requests Per Minute : {rpm:.2f}")

Requests Per Minute : 1.82


In [15]:
# ----------------------------------------------------
# Production Metric 8 : Token Usage
#
# Estimate prompt tokens and generated tokens.
#
# OpenAI provides this automatically.
# Local models require manual calculation.
# FLAN-T5 does not provide token usage like OpenAI APIs, but since we have the tokenizer we
# can compute it yourself.
# ----------------------------------------------------

response=generator(prompt)[0]["generated_text"]

prompt_tokens=len(tokenizer.encode(prompt))

completion_tokens=len(tokenizer.encode(response))

total_tokens=prompt_tokens+completion_tokens

print(f"Prompt Tokens      : {prompt_tokens}")

print(f"Completion Tokens  : {completion_tokens}")

print(f"Total Tokens       : {total_tokens}")

Prompt Tokens      : 15
Completion Tokens  : 101
Total Tokens       : 116


In [16]:
# ----------------------------------------------------
# Production Metric 9 : Token Generation Speed
#
# Measures how quickly the model generates tokens.
# ----------------------------------------------------

start=time.perf_counter()

response=generator(prompt)[0]["generated_text"]

elapsed=time.perf_counter()-start

tokens=len(tokenizer.encode(response))

print(f"Generated Tokens : {tokens}")

print(f"Elapsed Time : {elapsed:.2f} sec")

print(f"Generation Speed : {tokens/elapsed:.2f} tokens/sec")

Generated Tokens : 101
Elapsed Time : 36.95 sec
Generation Speed : 2.73 tokens/sec


In [17]:
# ----------------------------------------------------
# Production Metric 10 : GPU / CPU Memory Usage
#
# Helps detect memory leaks and oversized models.
# ----------------------------------------------------

import psutil
import os

process=psutil.Process(os.getpid())

memory_mb=process.memory_info().rss/(1024*1024)

print(f"Python Memory Usage : {memory_mb:.2f} MB")

Python Memory Usage : 3143.25 MB


### Switching to usage of bigger LLMs deployed on endpoints

In [26]:
#repeating previous steps
template2 = "Please write a {length} review,of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]

prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)

#Using AzureOpenAI and GPT model
#Note** If using gpt model and AzureOpenAI or AzureChatOpenAI (refer: 'Working_with_AzureOpenAI' files)
import openai
import os
from openai import AzureOpenAI

# Initialize client once
from dotenv import load_dotenv
#load_dotenv("/content/.env")
load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
)
deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")
#or



In [27]:
#Creating function (using Client based on AzureOpenAI or AzureChatOpenAI)
def get_completion(prompt, deployment_name=deployment_name):
    """Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )
        #return response.model_dump()  # Return the full response as dict
        # Extract just the assistant's reply
        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}

In [28]:
# def get_realtime_data(prompt):
#     print('realtime weather is good')

prompt = "what is the cryptography"

# if 'now' in prompt:
#     get_realtime_data(prompt)
# else:
get_completion(prompt)



'**Cryptography** is the science and art of securing information by transforming it into a form that only intended recipients can understand. It involves techniques for **encrypting** (scrambling) data to protect it from unauthorized access and **decrypting** (unscrambling) it so authorized users can read it.\n\n### Key Concepts in Cryptography\n\n- **Encryption:** Converting plain text (readable data) into cipher text (unreadable form) using an algorithm and a key.\n- **Decryption:** Reversing the encryption process to turn cipher text back into plain text.\n- **Key:** A piece of information (like a password or number) used in the encryption and decryption process.\n- **Cipher:** The algorithm or method used to perform encryption and decryption.\n\n### Types of Cryptography\n\n1. **Symmetric-key cryptography:** The same key is used for both encryption and decryption (e.g., AES, DES).\n2. **Asymmetric-key cryptography:** Uses a pair of keys—public and private. The public key encrypts, 

In [29]:
#For monitoring
#For xample Latency measurement

#Monitoring bigger LLM
# ----------------------------------------------------
# Production Metric 1 : End-to-End Latency
#
# Measures the total time taken to process one request.
# This is the response time experienced by the user.
# ----------------------------------------------------

import time

prompt = template2.format(
    length="short",
    book_title="Sherlock Holmes"
)

start = time.perf_counter()

response = get_completion(prompt)

end = time.perf_counter()

latency_ms = (end-start)*1000

print(response)

#if function contains - return response, then 
#print(response[0]["generated_text"])
print(f"\nLatency : {latency_ms:.2f} ms")

Certainly! Here’s a short review of *Sherlock Holmes*:

*Sherlock Holmes*, created by Sir Arthur Conan Doyle, is a timeless classic in detective fiction. The stories follow the brilliant and eccentric detective Sherlock Holmes and his loyal friend Dr. John Watson as they solve complex mysteries in Victorian London. Holmes’s keen powers of observation, logical reasoning, and unique methods make each case intriguing and suspenseful. The writing is sharp, atmospheric, and filled with memorable characters. Whether you’re reading *The Hound of the Baskervilles* or *A Study in Scarlet*, the adventures are consistently engaging. This collection is a must-read for anyone who enjoys clever mysteries and iconic literary figures.

Latency : 3034.81 ms


In [30]:
#Similarly other examples 2-10 as shown above.

Now we 've understood the manual monitoring. 
Now we can introduce LangSmith, showing how it automatically collects many of those metrics. 
Using Azure OpenAI (or any remote LLM), LangSmith fits naturally.

In [38]:
    # ----------------------------------------------------
# LangSmith Step 1 : Enable Tracing
#
# LangSmith automatically records every LLM call made
# through LangChain. Traces include prompts, responses,
# latency, errors, and execution hierarchy.
# ----------------------------------------------------

import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Your LangSmith API Key
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# Project name shown in the LangSmith UI
os.environ["LANGCHAIN_PROJECT"] = "flan-t5-demo"

print("LangSmith tracing enabled.")

'''This works automatically only for LangChain objects (e.g., AzureChatOpenAI). 
If we're calling the Azure OpenAI SDK directly with client.chat.completions.create(...), 
LangSmith will not automatically trace those calls. We'll have to use manual traces'''

LangSmith tracing enabled.


"This works automatically only for LangChain objects (e.g., AzureChatOpenAI). \nIf we're calling the Azure OpenAI SDK directly with client.chat.completions.create(...), \nLangSmith will not automatically trace those calls. We'll have to use manual traces"

In [32]:
#Manual tracing
# ----------------------------------------------------
# LangSmith Step 2 : Manual Trace
#
# Wrap the entire request in a LangSmith trace.
# This creates one parent run visible in LangSmith.
# ----------------------------------------------------

from langsmith import trace
import time

prompt = "Explain cryptography in simple words."

with trace(
    name="azure-chat-request",
    run_type="chain",
    tags=["azure-openai", "demo"],
    metadata={
        "model": deployment_name,
        "user": "demo_user"
    }
):

    start = time.perf_counter()

    response = get_completion(prompt)

    latency = (time.perf_counter() - start) * 1000

print(response)
print(f"Latency : {latency:.2f} ms")

Sure! Cryptography is like creating secret codes to keep information safe.

Imagine you want to send a message to your friend, but you don’t want anyone else to read it. You can scramble the message using a special rule (called a key), so only your friend, who knows the rule, can unscramble and read it. If someone else finds the message, it just looks like random letters and doesn’t make sense.

So, cryptography is the science of hiding information so only the right people can read it. It’s used in things like online banking, messaging apps, and websites to keep your data private and secure.
Latency : 2964.78 ms


In [33]:
#If we open LangSmith
#We see azure-chat-request with latency, metadata & tags.

In [34]:
#If Application (later) has multiple steps
# ----------------------------------------------------
# LangSmith Step 3 : Nested Traces
#
# Break the workflow into child spans so each step's
# execution time can be analyzed independently.
# ----------------------------------------------------

from langsmith import trace

prompt = "Explain blockchain."

with trace(name="qa-pipeline", run_type="chain"):

    with trace(name="generate-answer", run_type="llm"):

        response = get_completion(prompt)

print(response)

Certainly! Here’s a clear explanation:

**Blockchain** is a type of digital technology that allows information to be recorded in a secure, transparent, and tamper-proof way. Imagine it as a chain of blocks, where each block contains a list of transactions or data. These blocks are linked together in chronological order, forming a continuous chain—hence the name "blockchain."

**Key features of blockchain:**

1. **Decentralized:** Instead of being stored in one place (like a traditional database), copies of the blockchain are kept on many computers (called nodes) around the world. This means no single person or organization controls the entire chain.

2. **Secure and Transparent:** Once information is added to a block and the block is added to the chain, it’s very difficult to change. Everyone on the network can see the same data, making it transparent.

3. **Consensus Mechanism:** Before a new block is added, the network participants must agree that the transactions are valid. This is 

#LangSmith UI
qa-pipeline >  generate-answer
#Later (we can/would)
qa-pipeline > retrieve-documents, rerank, llm, summarize

In [35]:
#Add useful Metadata
# ----------------------------------------------------
# LangSmith Step 4 : Metadata
#
# Metadata provides searchable information about each
# request, such as model version, user, application
# version, or prompt version.
# ----------------------------------------------------

with trace(
    name="book-review",
    run_type="chain",
    metadata={
        "model": deployment_name,
        "prompt_version": "v1",
        "application": "Book Review Demo",
        "environment": "development"
    }
):

    response = get_completion(
        "Write a short review of Sherlock Holmes."
    )

print(response)

**Sherlock Holmes** is one of literature’s most iconic detectives, created by Sir Arthur Conan Doyle. Set in Victorian London, the stories follow Holmes’s brilliant deductive reasoning and his loyal companion, Dr. John Watson. The tales are gripping, filled with clever mysteries, atmospheric settings, and memorable characters. Holmes’s eccentric personality and razor-sharp intellect make him endlessly fascinating, while Watson’s narration adds warmth and humanity. The stories remain engaging and suspenseful, cementing Holmes’s legacy as the quintessential detective and a timeless figure in crime fiction.


Now we can filter in LangSmith by

-   model
-   prompt_version
-   environment

In [36]:
#Add tags : Tags help group similar requests
# ----------------------------------------------------
# LangSmith Step 5 : Tags
#
# Tags allow runs to be grouped and filtered in the
# LangSmith dashboard.
# ----------------------------------------------------

with trace(
    name="book-review",
    run_type="chain",
    tags=["review", "english", "demo"]
):

    response = get_completion(
        "Write a short review of Sherlock Holmes."
    )

print(response)

**Sherlock Holmes** is one of literature’s most iconic detectives, created by Sir Arthur Conan Doyle. Set in Victorian London, the stories follow Holmes’s brilliant deductive reasoning and his loyal companion, Dr. John Watson. The tales are gripping, filled with clever mysteries, atmospheric settings, and memorable characters. Holmes’s eccentric personality and razor-sharp intellect make him endlessly fascinating, while Watson’s narration adds warmth and humanity. The stories have stood the test of time, inspiring countless adaptations and remaining a benchmark for detective fiction. Whether you’re a mystery lover or a newcomer, Sherlock Holmes offers suspense, wit, and enduring appeal.


Then we can filter >
review, english, production,beta , premium-users

In [39]:
#Retrieve recent runs
# ----------------------------------------------------
# LangSmith Step 6 : Retrieve Recent Runs
#
# The LangSmith client can list runs that were
# previously recorded.
# ----------------------------------------------------

from langsmith import Client

client_ls = Client()

runs = list(
    client_ls.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        limit=5
    )
)

print(f"Found {len(runs)} runs\n")

for run in runs:

    print(f"Name      : {run.name}")
    print(f"Run ID    : {run.id}")
    print(f"Status    : {run.status}")
    print()

Found 5 runs

Name      : HuggingFacePipeline
Run ID    : 019fbf9b-f809-7a61-b3c9-51ba9fc447d5
Status    : success

Name      : step-2-summarise
Run ID    : df93d66b-773b-4471-92f2-01dae940dd3f
Status    : success

Name      : HuggingFacePipeline
Run ID    : 019fbf9b-e3e8-7763-b48c-bae4ba8a9e61
Status    : success

Name      : step-1-style
Run ID    : 38f44db5-4820-4d23-8793-7a626092bfbc
Status    : success

Name      : style-transfer-pipeline
Run ID    : 63db13b2-7d73-4ee0-8370-74f8573cbd1d
Status    : success



In [40]:
# ----------------------------------------------------
# LangSmith Step 7 : Aggregate Latency
#
# Calculate average latency using traces already stored
# in LangSmith.
# ----------------------------------------------------

import statistics

latencies = []

for run in runs:

    if run.start_time and run.end_time:

        latency = (
            run.end_time - run.start_time
        ).total_seconds() * 1000

        latencies.append(latency)

if latencies:

    print(f"Average Latency : {statistics.mean(latencies):.2f} ms")

Average Latency : 6138.84 ms


In [41]:
# ----------------------------------------------------
# LangSmith Step 8 : User Feedback
#
# Store user ratings for future evaluation and analysis.
# ----------------------------------------------------

latest = runs[0]

client_ls.create_feedback(

    run_id=latest.id,

    key="user_rating",

    score=1,

    comment="Helpful answer."
)

print("Feedback stored.")

Feedback stored.


In [42]:
# ----------------------------------------------------
# LangSmith Dashboard Summary
#
# The LangSmith dashboard now contains:
#
# ✓ Prompt
# ✓ Response
# ✓ Latency
# ✓ Run hierarchy
# ✓ Metadata
# ✓ Tags
# ✓ User feedback
#
# These traces can be filtered and analyzed to monitor
# application performance in production.
# ----------------------------------------------------

##### More examples to be added.